In [0]:
dbutils.widgets.text("entity_name", "products")
entity_name = dbutils.widgets.get("entity_name")

In [0]:
%run ../07_Common/00_setup

In [0]:
%run ../07_Common/Utils_Nombre_Archivos_Volumen

In [0]:
%run ../07_Common/02_bronze/Utils_Control_Config_Bronze


In [0]:
%run ../07_Common/02_bronze/Utils_Idempotencia_Bronze


In [0]:
%run ../07_Common/02_bronze/Utils_Watermark_Bronze

In [0]:
print(f"Leyendo metadata de {entity_name}")
bronze_config = get_bronze_config(entity_name)
raw_path = get_raw_path_for_entity(entity_name)

target_table     = bronze_config["target_table"]
load_mode        = bronze_config["load_mode"]
watermark_column = bronze_config["watermark_column"]
last_watermark   = bronze_config["last_watermark"]
execution_date   = date.today()

print(f"  Tabla destino: {target_table} | load_mode: {load_mode}")
if load_mode == "incremental":
    print(f"  Watermark actual: {last_watermark} | columna: {watermark_column}")

In [0]:

if bronze_already_loaded_today(target_table, execution_date):
    mensaje = f"OMITIDO | entidad='{entity_name}' | motivo='ya existe snapshot de hoy en {target_table}'"
    print(mensaje)
    dbutils.notebook.exit(mensaje)
    

In [0]:
file_name = build_raw_file_name(entity_name, execution_date)
full_raw_path = f"{raw_path}{file_name}"

if not path_exists(full_raw_path):
    mensaje = f"ERROR | entidad='{entity_name}' | motivo='no existe el archivo de Raw de hoy: {full_raw_path}'"
    print(mensaje)
    dbutils.notebook.exit(mensaje)

print(f"  Archivo de Raw encontrado: {full_raw_path}")

In [0]:
df_raw = spark.read.option("multiLine", "true").json(full_raw_path)

print(f"  Registros leídos: {df_raw.count()} | Columnas detectadas: {len(df_raw.columns)}")

In [0]:
df_bronze = (
    df_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_load_date", F.lit(execution_date))
    .withColumn("_source_file", F.lit(full_raw_path))
)

In [0]:
nuevo_watermark = None  # se mantiene None si load_mode == 'full'

if load_mode == "incremental":
    df_a_escribir = filter_by_watermark(df_bronze, watermark_column, last_watermark)
    registros_nuevos = df_a_escribir.count()

    print(f"   Registros con cambios desde el último watermark: {registros_nuevos}")

    if registros_nuevos == 0:
        mensaje = f"OMITIDO | entidad='{entity_name}' | motivo='sin registros nuevos desde last_watermark={last_watermark}'"
        print(mensaje)
        dbutils.notebook.exit(mensaje)

    nuevo_watermark = compute_new_watermark(df_a_escribir)
    print(f"   Nuevo watermark calculado: {nuevo_watermark}")

else:  # load_mode == "full"
    df_a_escribir = df_bronze
    print(f"  Carga FULL: se escriben los {df_a_escribir.count()} registros del snapshot completo")

In [0]:
try:
    if load_mode == "incremental":
        (
            df_a_escribir.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(target_table)
        )
    else:  
        (
            df_a_escribir.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")   
            .saveAsTable(target_table)
        )

    estado_final = "success"
    print(f"   Escritura exitosa en {target_table} (modo escritura: {'append' if load_mode == 'incremental' else 'overwrite'})")

except Exception as e:
    estado_final = "failed"
    print(f"   ERROR al escribir en Bronze: {e}")
    print("   Revisar si hubo un cambio de tipo incompatible en la API de origen.")

In [0]:
if load_mode == "incremental" and estado_final == "success" and nuevo_watermark is not None:
    spark.sql(f"""
        UPDATE workspace.control.bronze_load_config
        SET last_loaded_date = '{execution_date}',
            last_run_status  = '{estado_final}',
            last_watermark   = TIMESTAMP('{nuevo_watermark}')
        WHERE entity_name = '{entity_name}'
    """)
else:
    spark.sql(f"""
        UPDATE workspace.control.bronze_load_config
        SET last_loaded_date = '{execution_date}',
            last_run_status  = '{estado_final}'
        WHERE entity_name = '{entity_name}'
    """)

mensaje_final = f"{estado_final.upper()} | entidad='{entity_name}' | tabla='{target_table}' | modo='{load_mode}'"
print(mensaje_final)
dbutils.notebook.exit(mensaje_final)

In [0]:
df = spark.table("workspace.bronze.products")

In [0]:
df.display()